In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup
import pandas as pd
from pandas import ExcelWriter
import datetime
from time import sleep
import os

print("Running  Web Scraping Tool v.1.0")
scriptfolder=os.path.dirname(os.path.abspath(__file__))
os.chdir(scriptfolder)

now=datetime.datetime.now()
filename= 'BD CBBAN SQL Ready {}.xlsx'.format(str(now).replace(":",".")[:-7])
writer = ExcelWriter(filename)

driver = webdriver.Chrome()
driver.maximize_window()

regdict={'BD CBBAN 1': 'Banks', 
'BD CBBAN 2': 'Financial institutions', 
'BD CBBAN 3': 'Micro finance institutions', 
'BD CBBAN 4': 'Others'}


sqldict={'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 
		  'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 
		  'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 
		  'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],
		  'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 
		  'Phone - Mother company': [], 'Check': []}

processdate=now.strftime('%Y-%m-%d')

for reg in regdict:
	print(f'Working with {reg}.')
	driver.get('https://www.google.com')
	driver.get('https://www.bb.org.bd/links/index.php')
	sleep(5)
	driver.find_element(By.ID, 'rightmenuflat').find_element(By.LINK_TEXT, regdict[reg]).click()
	sleep(5)
	soup=BeautifulSoup(driver.page_source, 'html.parser')
	table=soup.find('table', {'class': 'standardcellborder'}).find('tbody')
	for tr in table.find_all('tr')[1:]:#header in index 0
		print(tr.text)
		sqldict['Name'].append(tr.find('td').text.strip())
		if tr.find('a', href=True):#in case of missing anchors/links
			sqldict['Website'].append(tr.find('a', href=True)['href'])
		sqldict['ListProcessDate'].append(processdate)
		sqldict['Cntry'].append('BD')
		sqldict['RegCtry'].append('BD')
		sqldict['RegCode'].append('CBBAN')
		sqldict['ListCode'].append(reg.split(' ')[-1])
		sqldict['RegulationType'].append('Regulated')
		for key in sqldict.keys():
		 	while len(sqldict[key])<len(sqldict['ListProcessDate']):
		 		sqldict[key].append('')
os.chdir(scriptfolder)
#for key in sqldict.keys():
#	print(key, ': ', len(sqldict[key]))

df=pd.DataFrame(sqldict)
df.to_excel(writer, 'SQL ready', index=False)
writer.save()
writer.close()
sleep(3)

driver.quit()

endtime=datetime.datetime.now()
difference=endtime-now
difference=difference.total_seconds()
file = open(filename.replace('data', 'time').replace('xlsx','txt'),'w') 
file.write("Start: {} \nEnd: {} \nTotal: {} hours, {} minutes and {} seconds.".format(str(now)[:-7], str(endtime)[:-7], int(difference//3600),int(difference%3600)//60,int(difference%3600)%60))
file.close()
    
    